In [ ]:
# Install CLIP
!pip install ftfy regex tqdm
!pip install openai_clip

In [ ]:
# Import necessary libraries
import torch
import torch.nn as nn
import torch.nn.functional as F
import clip
import numpy as np
from torch.utils.data import DataLoader
import os
from PIL import Image

In [ ]:
class PromptLearner(nn.Module):
    """Learns context tokens similar to CoOp but with regularization"""
    
    def __init__(self, clip_model, classnames, n_ctx=8, reg_weight=0.1):
        super().__init__()
        
        self.n_ctx = n_ctx # number of learned context tokens
        self.reg_weight = reg_weight # regularization weight
        self.classnames = classnames
        self.C = len(classnames) # number of classes
        self.dtype = clip_model.dtype
        self.device = next(clip_model.parameters()).device
        self.clip_model = clip_model
        self.text_encoder = TextEncoder(clip_model)
        
        # Get context dimension
        D = clip_model.ln_final.weight.shape[0]
        
        # Initialize learnable context vectors
        ctx_vectors = torch.empty(n_ctx, D, dtype=self.dtype) # [n_ctx, D]
        nn.init.normal_(ctx_vectors, std=0.02)
        self.ctx = nn.Parameter(ctx_vectors)
        
        # Store regularization target
        self.register_buffer('reg_target', self._create_regularization_target(clip_model))

        # Store avg class name embedding
        self.register_buffer('class_embeddings', self._embed_class_names(clip_model)) 
        
    def _create_regularization_target(self):
        template = "a photo of a {}, a type of flower"
        prompts = [template.format(classname) for classname in self.classnames] # list of C prompts, where C is the number of classes
        
        tokenized = clip.tokenize(prompts).to(self.device) # [C, 77], where 77 is CLIP's number of sequence tokens
        
        with torch.no_grad():
            text_features = self.clip_model.encode_text(tokenized) # [C, D]
            avg_features = text_features.mean(dim=0)  # [D]
            
        return avg_features
    
    def _embed_class_names(self):
        tokenized = clip.tokenize(self.classnames).to(self.device) # [C, 77]
        embeddings = self.clip_model.token_embedding(tokenized) # [C, 77, D]
        # mask out pad, SOT and EOT 
        mask = (tokenized != 0) & (tokenized != 49406) & (tokenized != 49407) # [C, 77]
        class_embeddings = []
        for i in range(len(self.classnames)):
            valid_positions = mask[i].nonzero(as_tuple=True)[0] # get indices where mask is True
            valid_embeddings = embeddings[i, valid_positions, :]
            class_embeddings.append(valid_embeddings)
        
        return class_embeddings # list of C tensors, each of shape [L_c, D] where L_c is the number of tokens for the class name and it's variable from class to class
    
    def compute_regularization_loss(self):
        """Compute regularization loss using cosine similarity"""
        learned_avg = self.ctx.mean(dim=0)
        
        # Normalize vectors for cosine similarity
        learned_norm = F.normalize(learned_avg, dim=0)
        target_norm = F.normalize(self.reg_target, dim=0)
        
        # Cosine similarity loss (we want to maximize similarity, so minimize 1 - cosine)
        cosine_sim = torch.dot(learned_norm, target_norm)
        reg_loss = 1.0 - cosine_sim
        
        return reg_loss

    
    def forward(self):
        prompts = []
        for c in self.class_embeddings:
            full_prompt = torch.cat([self.ctx, c], dim=0)
            prompts.append(full_prompt)
        return prompts



class ConditionalPromptLearner(nn.Module):
    """Learns instance-conditioned context tokens via MLP"""
    
    def __init__(self, clip_model, n_ctx=4):
        super().__init__()
        
        self.n_ctx = n_ctx
        self.dtype = clip_model.dtype
        
        # Get dimensions
        vis_dim = clip_model.visual.output_dim
        ctx_dim = clip_model.ln_final.weight.shape[0]
        
        # MLP for instance conditioning
        self.mlp = nn.Sequential(
            nn.Linear(vis_dim, ctx_dim),
            nn.ReLU(),
            nn.Linear(ctx_dim, n_ctx * ctx_dim)
        )
        
    def forward(self, image_features):
        """Generate instance-conditioned context tokens"""
        batch_size = image_features.shape[0]
        
        # Generate tokens via MLP
        ctx_logits = self.mlp(image_features.float())
        ctx_tokens = ctx_logits.view(batch_size, self.n_ctx, -1).type(self.dtype)
        
        return ctx_tokens


class TextEncoder(nn.Module):
    """Custom text encoder for multiple prompt configurations"""
    
    def __init__(self, clip_model, classnames):
        super().__init__()
        
        self.clip_model = clip_model
        self.classnames = classnames
        self.dtype = clip_model.dtype
        self.device = next(clip_model.parameters()).device
        
        # CLIP text encoder components
        self.token_embedding = clip_model.token_embedding
        self.transformer = clip_model.transformer
        self.positional_embedding = clip_model.positional_embedding
        self.ln_final = clip_model.ln_final
        self.text_projection = clip_model.text_projection
        
        # Tokenize class names
        self.register_buffer('tokenized_prompts', self._tokenize_classnames())
        
    def _tokenize_classnames(self):
        """Tokenize class names for prompt construction"""
        prompts = [f"X. {classname.replace('_', ' ')}" for classname in self.classnames]
        return clip.tokenize(prompts)
    
    def construct_prompts(self, learned_ctx, instance_ctx, batch_size):
        """
        Construct three prompt configurations:
        1. [IC, L, CLS] - Instance + Learned + Class
        2. [L, IC, CLS] - Learned + Instance + Class
        3. [L, CLS, IC] - Learned + Class + Instance
        """
        n_cls = len(self.classnames)
        
        # Get token embeddings for class names
        tokenized = self.tokenized_prompts.expand(batch_size, -1, -1)
        tokenized = tokenized.contiguous().view(-1, tokenized.shape[-1])
        token_embeddings = self.token_embedding(tokenized).type(self.dtype)
        
        # Extract components
        prefix = token_embeddings[:, :1, :]  # [SOS] token
        suffix = token_embeddings[:, 2:, :]  # Class name + [EOS] + padding
        
        # Expand contexts for all classes
        learned_expanded = learned_ctx.unsqueeze(1).expand(-1, n_cls, -1, -1)
        learned_expanded = learned_expanded.contiguous().view(batch_size * n_cls, -1, learned_expanded.shape[-1])
        
        instance_expanded = instance_ctx.unsqueeze(1).expand(-1, n_cls, -1, -1)
        instance_expanded = instance_expanded.contiguous().view(batch_size * n_cls, -1, instance_expanded.shape[-1])
        
        # Create three configurations
        prompts_1 = torch.cat([prefix, instance_expanded, learned_expanded, suffix], dim=1)
        prompts_2 = torch.cat([prefix, learned_expanded, instance_expanded, suffix], dim=1)
        
        # For config 3, insert instance tokens before suffix
        class_tokens = suffix[:, :1, :]
        remaining_suffix = suffix[:, 1:, :]
        prompts_3 = torch.cat([prefix, learned_expanded, class_tokens, instance_expanded, remaining_suffix], dim=1)
        
        return prompts_1, prompts_2, prompts_3
    
    def encode_prompts(self, prompts):
        """Encode prompts through CLIP text encoder"""
        # Add positional embeddings
        x = prompts + self.positional_embedding[:prompts.shape[1], :].type(self.dtype)
        x = x.permute(1, 0, 2)  # NLD -> LND
        x = self.transformer(x)
        x = x.permute(1, 0, 2)  # LND -> NLD
        x = self.ln_final(x).type(self.dtype)
        
        # Extract features at end-of-text position
        eot_pos = prompts.shape[1] - 1
        text_features = x[:, eot_pos, :] @ self.text_projection
        
        return text_features
    
    def forward(self, learned_ctx, instance_ctx, batch_size):
        """Forward pass through text encoder"""
        # Construct prompts
        prompts_1, prompts_2, prompts_3 = self.construct_prompts(learned_ctx, instance_ctx, batch_size)
        
        # Encode all configurations
        features_1 = self.encode_prompts(prompts_1)
        features_2 = self.encode_prompts(prompts_2)
        features_3 = self.encode_prompts(prompts_3)
        
        # Reshape and average
        n_cls = len(self.classnames)
        features_1 = features_1.view(batch_size, n_cls, -1)
        features_2 = features_2.view(batch_size, n_cls, -1)
        features_3 = features_3.view(batch_size, n_cls, -1)
        
        # Average the three configurations
        text_features = (features_1 + features_2 + features_3) / 3.0
        
        return text_features


class CustomCLIP(nn.Module):
    """
    Wrapper class for the custom method combining:
    - PromptLearner: Regularized learned context tokens
    - ConditionalPromptLearner: Instance-conditioned tokens via MLP
    - TextEncoder: Multiple prompt configurations with ensemble
    """
    
    def __init__(self, clip_model, classnames, n_learned_ctx=8, n_instance_ctx=4, reg_weight=0.1):
        super().__init__()
        
        self.clip_model = clip_model
        self.classnames = classnames
        self.n_learned_ctx = n_learned_ctx
        self.n_instance_ctx = n_instance_ctx
        self.reg_weight = reg_weight
        
        # Initialize components
        self.prompt_learner = PromptLearner(clip_model, classnames, n_learned_ctx, reg_weight)
        self.conditional_prompt_learner = ConditionalPromptLearner(clip_model, n_instance_ctx)
        self.text_encoder = TextEncoder(clip_model, classnames)
        
    def forward(self, image, image_features=None):
        """Forward pass"""
        batch_size = image.shape[0]
        
        # Get image features if not provided
        if image_features is None:
            image_features = self.clip_model.encode_image(image)
        
        # Get context tokens
        learned_ctx = self.prompt_learner(batch_size)
        instance_ctx = self.conditional_prompt_learner(image_features)
        
        # Encode text with multiple configurations
        text_features = self.text_encoder(learned_ctx, instance_ctx, batch_size)
        
        # Normalize features
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        
        # Compute similarity logits
        logits = torch.einsum('bd,bcd->bc', image_features, text_features) * self.clip_model.logit_scale.exp()
        
        return logits
    
    def compute_loss(self, logits, targets):
        """Compute total loss including cross-entropy and regularization"""
        ce_loss = F.cross_entropy(logits, targets)
        reg_loss = self.prompt_learner.compute_regularization_loss()
        
        total_loss = ce_loss + self.reg_weight * reg_loss
        
        return total_loss, ce_loss, reg_loss

In [ ]:
def load_flowers102_classnames():
    """Load Flowers102 class names"""
    # Flowers102 class names (you might need to adjust this based on your dataset)
    flowers_classes = [
        "pink primrose", "hard-leaved pocket orchid", "canterbury bells", "sweet pea", "wild geranium",
        "tiger lily", "moon orchid", "bird of paradise", "monkshood", "globe thistle",
        "snapdragon", "colt's foot", "king protea", "spear thistle", "yellow iris",
        "globe-flower", "purple coneflower", "peruvian lily", "balloon flower", "giant white arum lily",
        "fire lily", "pincushion flower", "fritillary", "red ginger", "grape hyacinth",
        "corn poppy", "prince of wales feathers", "stemless gentian", "artichoke", "sweet william",
        "carnation", "garden phlox", "love in the mist", "cosmos", "alpine sea holly",
        "ruby-lipped cattleya", "cape flower", "great masterwort", "siam tulip", "lenten rose",
        "barberton daisy", "daffodil", "sword lily", "poinsettia", "bolero deep blue",
        "wallflower", "marigold", "buttercup", "daisy", "common dandelion",
        "petunia", "wild pansy", "primula", "sunflower", "lilac hibiscus",
        "bishop of llandaff", "gaura", "geranium", "orange dahlia", "pink-yellow dahlia",
        "cautleya spicata", "japanese anemone", "black-eyed susan", "silverbush", "californian poppy",
        "osteospermum", "spring crocus", "iris", "windflower", "tree poppy",
        "gazania", "azalea", "water lily", "rose", "thorn apple",
        "morning glory", "passion flower", "lotus", "toad lily", "anthurium",
        "frangipani", "clematis", "hibiscus", "columbine", "desert-rose",
        "tree mallow", "magnolia", "cyclamen", "watercress", "canna lily",
        "hippeastrum", "bee balm", "pink quill", "foxglove", "bougainvillea",
        "camellia", "mallow", "mexican petunia", "bromelia", "blanket flower",
        "trumpet creeper", "blackberry lily", "common tulip", "wild rose"
    ]
    
    return flowers_classes

def create_simple_dataset(image_paths, labels, transform=None):
    """Create a simple dataset for testing"""
    class SimpleDataset(torch.utils.data.Dataset):
        def __init__(self, image_paths, labels, transform=None):
            self.image_paths = image_paths
            self.labels = labels
            self.transform = transform
            
        def __len__(self):
            return len(self.image_paths)
        
        def __getitem__(self, idx):
            image = Image.open(self.image_paths[idx]).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, self.labels[idx]
    
    return SimpleDataset(image_paths, labels, transform)

In [ ]:
# Example usage and testing
def test_custom_method():
    """Test the custom method implementation"""
    
    # Load CLIP model
    device = "cuda" if torch.cuda.is_available() else "cpu"
    clip_model, preprocess = clip.load("ViT-B/32", device=device)
    
    # Load class names
    classnames = load_flowers102_classnames()
    print(f"Loaded {len(classnames)} flower classes")
    
    # Initialize custom method
    model = CustomCLIP(
        clip_model=clip_model,
        classnames=classnames,
        n_learned_ctx=8,
        n_instance_ctx=4,
        reg_weight=0.1
    ).to(device)
    
    print(f"Model initialized with:")
    print(f"- {model.n_learned_ctx} learned context tokens")
    print(f"- {model.n_instance_ctx} instance-conditioned context tokens")
    print(f"- Regularization weight: {model.reg_weight}")
    
    # Test with dummy data
    batch_size = 4
    dummy_images = torch.randn(batch_size, 3, 224, 224).to(device)
    dummy_labels = torch.randint(0, len(classnames), (batch_size,)).to(device)
    
    print(f"\nTesting with dummy batch of size {batch_size}")
    
    # Forward pass
    with torch.no_grad():
        logits = model(dummy_images)
        print(f"Output logits shape: {logits.shape}")
        
        # Compute loss
        total_loss, ce_loss, reg_loss = model.compute_loss(logits, dummy_labels)
        print(f"Cross-entropy loss: {ce_loss.item():.4f}")
        print(f"Regularization loss: {reg_loss.item():.4f}")
        print(f"Total loss: {total_loss.item():.4f}")
        
        # Check predictions
        predictions = logits.argmax(dim=1)
        print(f"Predictions: {predictions.cpu().numpy()}")
        print(f"True labels: {dummy_labels.cpu().numpy()}")
    
    return model

# Run the test
print("Testing Custom Method Implementation...")
model = test_custom_method()

In [ ]:
def train_custom_method(model, train_loader, val_loader=None, epochs=10, lr=0.002):
    """
    Training function for the custom method
    """
    optimizer = torch.optim.Adam([
        {'params': model.prompt_learner.parameters(), 'lr': lr},
        {'params': model.conditional_prompt_learner.parameters(), 'lr': lr * 0.1}  # Lower LR for MLP
    ])
    
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    
    model.train()
    device = next(model.parameters()).device
    
    for epoch in range(epochs):
        total_loss = 0
        total_ce_loss = 0
        total_reg_loss = 0
        correct = 0
        total = 0
        
        for batch_idx, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            
            # Forward pass
            logits = model(images)
            
            # Compute loss
            loss, ce_loss, reg_loss = model.compute_loss(logits, labels)
            
            # Backward pass
            loss.backward()
            optimizer.step()
            
            # Statistics
            total_loss += loss.item()
            total_ce_loss += ce_loss.item()
            total_reg_loss += reg_loss.item()
            
            _, predicted = logits.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            if batch_idx % 10 == 0:
                print(f'Epoch: {epoch+1}/{epochs} [{batch_idx * len(images)}/{len(train_loader.dataset)} '
                      f'({100. * batch_idx / len(train_loader):.0f}%)]\\t'
                      f'Loss: {loss.item():.6f} (CE: {ce_loss.item():.4f}, Reg: {reg_loss.item():.4f})')
        
        scheduler.step()
        
        # Epoch summary
        avg_loss = total_loss / len(train_loader)
        avg_ce_loss = total_ce_loss / len(train_loader)
        avg_reg_loss = total_reg_loss / len(train_loader)
        accuracy = 100. * correct / total
        
        print(f'\\nEpoch {epoch+1}/{epochs} Summary:')
        print(f'Train Loss: {avg_loss:.4f} (CE: {avg_ce_loss:.4f}, Reg: {avg_reg_loss:.4f})')
        print(f'Train Accuracy: {accuracy:.2f}%')
        
        # Validation
        if val_loader is not None:
            val_acc = evaluate_custom_method(model, val_loader)
            print(f'Val Accuracy: {val_acc:.2f}%')
        
        print('-' * 80)
    
    return model

def evaluate_custom_method(model, test_loader):
    """
    Evaluation function for the custom method
    """
    model.eval()
    device = next(model.parameters()).device
    
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            
            logits = model(images)
            _, predicted = logits.max(1)
            
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    accuracy = 100. * correct / total
    return accuracy

print("Training and evaluation functions defined!")

# Custom Method for Few-Shot Learning

## Overview

This notebook implements a custom method for few-shot base-to-novel adaptation on the Flowers102 dataset. The method combines several innovative techniques:

### Key Components

1. **Regularized Learned Prompts**: Like CoOp, we learn context tokens, but we regularize them to be similar to the average embedding of "a photo of a {class_name}, a type of flower" across all classes.

2. **Instance-Conditioned Context Tokens**: We use a simple MLP that takes CLIP image features as input and outputs embeddings for 4 context tokens that are specific to each input image.

3. **Multiple Prompt Configurations**: We create three different prompt arrangements and average their predictions:
   - `[IC, L, CLS]`: Instance-conditioned + Learned + Class tokens
   - `[L, IC, CLS]`: Learned + Instance-conditioned + Class tokens  
   - `[L, CLS, IC]`: Learned + Class + Instance-conditioned tokens

### Architecture Details

- **Learned Context Tokens**: 8 tokens (default), initialized with normal distribution
- **Instance MLP**: Linear layer with ReLU activation (image_features → context_embeddings)
- **Regularization Weight**: 0.1 (default)
- **Loss Function**: Cross-entropy + regularization loss

### Usage

1. Initialize the model with CLIP backbone and class names
2. Train using the provided training function
3. Evaluate on test data

The method should be particularly effective for few-shot scenarios where we have limited training data but want to leverage both learned and instance-specific representations.